# Rate-Limited API Service -- notebook demo

This notebook is a companion to the course's [Build a Rate-Limited API Service](https://abderrahim-lectures.github.io/python-data-analysis-course/docs/projects/rate-limited-api) lesson.

Most projects in this series that build a long-running server (like the [MCP Server](https://abderrahim-lectures.github.io/python-data-analysis-course/docs/projects/mcp-server) project) are a poor fit for a notebook -- there's no real port to connect to from Colab or Kaggle. This one is different: FastAPI's own `TestClient` drives the app **in-process**, with no real network socket or port at all, so the whole API -- pagination, filtering, API-key auth, and rate limiting -- can be exercised directly inside notebook cells. This is a good way to *see the behavior*; it is not a substitute for actually running `uvicorn` and hitting it with real HTTP requests, which the lesson's Setup and Steps walk through separately.

Run the cells in order.

In [ ]:
!pip install -q fastapi "httpx>=0.27"

## Get the app's source

Colab/Kaggle start from a blank filesystem, so pull `main.py`, `quotes_data.py`, and `rate_limit.py` straight from the course repo instead of assuming they're already sitting next to this notebook (they will be, if you cloned the whole repo instead).

In [ ]:
import os
import urllib.request

RAW_BASE = "https://raw.githubusercontent.com/abderrahim-lectures/python-data-analysis-course/main/examples/rate-limited-api"

if not os.path.exists("main.py"):
    for filename in ["main.py", "quotes_data.py", "rate_limit.py"]:
        urllib.request.urlretrieve(f"{RAW_BASE}/{filename}", filename)
    print("Downloaded main.py, quotes_data.py, rate_limit.py")
else:
    print("Files already present (running from a full repo checkout).")

## Build a `TestClient` -- no real server, no real port

`TestClient` wraps the FastAPI `app` object directly and sends requests to it through Python function calls, not sockets. That's what makes this notebook-friendly: it behaves exactly like the real HTTP API (same routes, same status codes, same headers) without needing `uvicorn` running in the background or a port forwarded out of the notebook's sandbox.

In [ ]:
from fastapi.testclient import TestClient
from main import app

client = TestClient(app)
print("TestClient ready -- talking directly to the FastAPI app object, in-process.")

## Pagination and filtering

In [ ]:
response = client.get("/quotes", params={"limit": 3})
print(response.status_code)
response.json()

In [ ]:
response = client.get("/quotes", params={"category": "science", "limit": 5})
print(response.status_code, "total matching:", response.json()["total"])
for quote in response.json()["items"]:
    print(f"- {quote['author']}: {quote['text']}")

In [ ]:
response = client.get("/quotes", params={"author": "sagan"})
response.json()

## Auth: a real 401 with no key, then a real key

In [ ]:
response = client.get("/me")
print(response.status_code, response.json())
assert response.status_code == 401

In [ ]:
response = client.post("/keys")
api_key = response.json()["api_key"]
print("Issued key:", api_key)

In [ ]:
response = client.get("/me", headers={"X-API-Key": api_key})
print(response.status_code, response.headers.get("x-ratelimit-limit"), response.json())
assert response.status_code == 200

## Rate limiting: trigger a real 429

`main.py` sets the limit to 5 requests per 10-second window. Firing 6 requests back-to-back in one cell -- with no `time.sleep` between them -- reliably lands inside that window and trips the limiter on the 6th call.

In [ ]:
fresh_key = client.post("/keys").json()["api_key"]

for attempt in range(1, 7):
    response = client.get("/me", headers={"X-API-Key": fresh_key})
    print(f"request {attempt}: {response.status_code}", response.headers.get("retry-after", ""))

assert response.status_code == 429
assert "retry-after" in response.headers
print("\nFinal response body:", response.json())

The 6th request comes back `429 Too Many Requests` with a real `Retry-After` header telling the client how long to wait -- driven entirely by `rate_limit.py`'s `SlidingWindowRateLimiter`, with no external rate-limiting service involved. See the lesson itself for why this in-memory approach resets on restart and doesn't share state across multiple server processes, and what you'd reach for in production instead.